In [ ]:

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# 1. Try Modern Small Models
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
model = model.to(device)

print(f"{model_name} loaded successfully!")

In [ ]:
# 2. Chat Format (Very Important)
def generate_response(prompt, max_new_tokens=300, temperature=0.7):
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test
print(generate_response("Tell me a short funny story about an Ethiopian programmer."))

In [ ]:
# 3. Mini Project: AI Assistant
print("=== AI Assistant ===")

def ai_assistant(query):
    prompt = f"""You are a helpful AI assistant. Answer the following question clearly and concisely.

Question: {query}"""
    
    return generate_response(prompt, max_new_tokens=250, temperature=0.6)

# Test the assistant
questions = [
    "What are the best career options in AI in 2026?",
    "Give me 5 Ethiopian food recipes that are easy to make.",
    "Explain transformers in simple terms."
]

for q in questions:
    print(f"\nQ: {q}")
    print("A:", ai_assistant(q)[:400] + "...")